# FashionCLIP 추구미 분류 실험

`configs/aesthetics.yaml`의 추구미를 FashionCLIP zero-shot으로 구분할 수 있는지 확인한다.

- 1단계(`PROMPT_SET = "outfit"`): 코디 사진으로 추구미가 구분되는지 확인한다.
- 2단계(`PROMPT_SET = "item"`): 옷 단독 사진으로 실제 서비스 조건을 확인한다. yaml의 `prompts.item`을 채운 뒤 실행한다.

**실행 방법 (Colab, GPU 런타임)**

1. 아래 셀의 주석을 풀어 레포를 받는다.
2. 이미지를 `data/aesthetics/<PROMPT_SET>/<추구미 key>/`에 넣는다. 예: `data/aesthetics/outfit/y2k/001.jpg`
3. 이미지가 아직 없으면 `SMOKE_TEST = True`로 실행 흐름만 확인한다. 이때 정확도와 점수는 의미가 없다.

결과는 `outputs/aesthetic_clip/<PROMPT_SET>/`에 저장된다. `data/`와 `outputs/`는 커밋되지 않는다.

In [ ]:
# Colab에서만 주석을 풀어 실행한다.
# !git clone -b develop https://github.com/My-Fit-Core/My-fit-core-ai.git
# %cd My-fit-core-ai

In [ ]:
from pathlib import Path

MODEL_ID = "patrickjohncyh/fashion-clip"
PROMPT_SET = "outfit"  # outfit: 코디 사진, item: 옷 단독 사진
SMOKE_TEST = False  # True면 단색 더미 이미지로 실행 흐름만 확인한다
MAX_IMAGES_PER_CLASS = None  # 몇 장만 돌려볼 때 정수로 지정한다
BATCH_SIZE = 16

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/aesthetics.yaml").exists()), None)
if ROOT is None:
    raise FileNotFoundError("configs/aesthetics.yaml을 찾지 못했습니다. 레포 안에서 실행하세요.")
DATA_DIR = ROOT / "data/aesthetics" / PROMPT_SET
OUTPUT_DIR = ROOT / "outputs/aesthetic_clip" / PROMPT_SET

In [ ]:
import yaml

config = yaml.safe_load((ROOT / "configs/aesthetics.yaml").read_text(encoding="utf-8"))
aesthetics = config["aesthetics"]
keys = [a["key"] for a in aesthetics]

missing = [a["key"] for a in aesthetics if not a["prompts"][PROMPT_SET]]
if missing:
    raise ValueError(f"prompts.{PROMPT_SET}이 비어 있습니다: {missing}")

# 프롬프트를 한 줄로 펼치고, 추구미별로 자기 프롬프트 위치를 기억한다.
prompts, prompt_ids_by_class = [], []
for a in aesthetics:
    class_prompts = a["prompts"][PROMPT_SET]
    prompt_ids_by_class.append(list(range(len(prompts), len(prompts) + len(class_prompts))))
    prompts.extend(class_prompts)

print(f"추구미 {len(keys)}개, {PROMPT_SET} 프롬프트 {len(prompts)}개: {keys}")

In [ ]:
import tempfile

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png"}


def make_smoke_images(class_keys, per_class=2):
    from PIL import Image

    root = Path(tempfile.mkdtemp())
    for c, key in enumerate(class_keys):
        (root / key).mkdir()
        for i in range(per_class):
            Image.new("RGB", (512, 512), (40 * c, 200 - 40 * c, 60 * i)).save(root / key / f"{i}.png")
    return root


def collect_images(data_dir, class_keys, limit=None):
    samples = []
    for key in class_keys:
        paths = sorted(p for p in (data_dir / key).glob("*") if p.suffix.lower() in IMAGE_SUFFIXES)
        samples += [(p, key) for p in paths[:limit]]
    return samples


source_dir = make_smoke_images(keys) if SMOKE_TEST else DATA_DIR
samples = collect_images(source_dir, keys, MAX_IMAGES_PER_CLASS)
if not samples:
    raise FileNotFoundError(f"이미지가 없습니다. {DATA_DIR}/<key>/ 에 넣거나 SMOKE_TEST = True로 실행하세요. key: {keys}")
for key in keys:
    print(f"{key}: {sum(k == key for _, k in samples)}장")

## 점수 계산

추구미별로 프롬프트 점수(logit)를 평균한 뒤 softmax를 적용한다. 점수는 이미지마다 합이 1인 상대값이라, 추구미 목록이나 프롬프트가 바뀌면 값도 달라진다.

softmax 전 코사인 유사도(프롬프트 평균)도 같이 저장한다. 모든 추구미와 코사인이 낮은 이미지는 어느 추구미에도 맞지 않는 이미지 후보다.

In [ ]:
import torch
from PIL import Image, ImageOps
from transformers import CLIPModel, CLIPProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
model = CLIPModel.from_pretrained(MODEL_ID).to(device).eval()
processor = CLIPProcessor.from_pretrained(MODEL_ID)


@torch.no_grad()
def score_batch(paths):
    images = [ImageOps.exif_transpose(Image.open(p)).convert("RGB") for p in paths]
    inputs = processor(text=prompts, images=images, return_tensors="pt", padding=True, truncation=True).to(device)
    logits = model(**inputs).logits_per_image  # (이미지 수, 프롬프트 수)
    per_class = torch.stack([logits[:, ids].mean(dim=1) for ids in prompt_ids_by_class], dim=1)
    cosines = per_class / model.logit_scale.exp()  # logit = logit_scale * 코사인
    return per_class.softmax(dim=1).cpu().tolist(), cosines.cpu().tolist()


paths = [p for p, _ in samples]
scores, cosines = [], []
for start in range(0, len(paths), BATCH_SIZE):
    batch_scores, batch_cosines = score_batch(paths[start : start + BATCH_SIZE])
    scores += batch_scores
    cosines += batch_cosines
print(f"device={device}, 이미지 {len(scores)}장 처리")

## 정확도, 혼동 행렬, 정답별 평균 점수

In [ ]:
import csv


def evaluate(labels, score_rows, class_keys):
    """정답 key 목록과 이미지별 점수로 예측, 정확도, 혼동 행렬(행=정답, 열=예측)을 계산한다."""
    preds = [class_keys[row.index(max(row))] for row in score_rows]
    matrix = [[0] * len(class_keys) for _ in class_keys]
    for label, pred in zip(labels, preds):
        matrix[class_keys.index(label)][class_keys.index(pred)] += 1
    accuracy = sum(lab == pred for lab, pred in zip(labels, preds)) / len(labels)
    return preds, accuracy, matrix


def mean_scores_by_label(labels, score_rows, class_keys):
    """정답 추구미별 평균 점수. 이미지가 없는 추구미는 결과에서 뺀다."""
    means = {}
    for key in class_keys:
        rows = [row for lab, row in zip(labels, score_rows) if lab == key]
        if rows:
            means[key] = [sum(col) / len(rows) for col in zip(*rows)]
    return means


def second_best(row, class_keys):
    return class_keys[sorted(range(len(row)), key=row.__getitem__)[-2]]


labels = [k for _, k in samples]
preds, accuracy, matrix = evaluate(labels, scores, keys)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_DIR / "scores.csv", "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow(["image", "label", "pred", "pred_2nd", *keys, *(f"cos_{k}" for k in keys)])
    for (path, label), pred, row, cos_row in zip(samples, preds, scores, cosines):
        values = [f"{v:.4f}" for v in (*row, *cos_row)]
        writer.writerow([path.name, label, pred, second_best(row, keys), *values])

if SMOKE_TEST:
    print("SMOKE_TEST: 더미 이미지 결과이므로 정확도와 점수에 의미가 없습니다.")
print(f"정확도: {accuracy:.3f} ({len(labels)}장, 무작위 기준 {1 / len(keys):.3f})")

width = max(len(k) for k in keys) + 1
header = " " * width + "".join(k.rjust(width) for k in keys)
print("\n혼동 행렬 (행=정답, 열=예측)")
print(header)
for key, row in zip(keys, matrix):
    print(key.ljust(width) + "".join(str(n).rjust(width) for n in row))

print("\n정답별 평균 점수 (행=정답, 열=추구미)")
print(header)
for key, row in mean_scores_by_label(labels, scores, keys).items():
    print(key.ljust(width) + "".join(f"{s:.3f}".rjust(width) for s in row))

print(f"\n이미지별 점수: {OUTPUT_DIR / 'scores.csv'}")

## 이미지별 점수 분포

- 히트맵: 이미지 한 장이 어느 추구미에 걸쳐 있는지 본다. 한 줄에 진한 칸이 여러 개면 여러 추구미에 걸친 이미지다.
- 히스토그램: 추구미별로 점수가 어떻게 퍼져 있는지 본다.

이미지별 정확한 값은 `scores.csv`에 있다(추구미 key 열은 softmax 점수, `cos_<key>` 열은 코사인 유사도). 차트 라벨은 Colab 기본 글꼴에서 한글이 깨지므로 영어로 쓴다.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

MAX_IMAGE_LABELS = 60  # 이보다 많으면 히트맵의 이미지 이름을 생략한다

order = sorted(range(len(samples)), key=lambda i: keys.index(labels[i]))
score_array = np.array(scores)

fig, ax = plt.subplots(figsize=(1.2 * len(keys) + 3, max(3, 0.28 * len(order))))
image = ax.imshow(score_array[order], cmap="Blues", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(keys)), keys, rotation=30, ha="right")
if len(order) <= MAX_IMAGE_LABELS:
    ax.set_yticks(range(len(order)), [f"{labels[i]}/{samples[i][0].name}" for i in order], fontsize=8)
else:
    ax.set_yticks([])
    ax.set_ylabel("images (sorted by label)")
ax.set_title("Aesthetic score per image")
fig.colorbar(image, ax=ax, label="score")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "score_heatmap.png", dpi=150)
plt.show()

fig, axes = plt.subplots(1, len(keys), figsize=(2.4 * len(keys), 2.6), sharex=True, sharey=True, squeeze=False)
for c, (key, ax) in enumerate(zip(keys, axes[0])):
    ax.hist(score_array[:, c], bins=10, range=(0, 1), color="#2b6cb0", rwidth=0.9)
    ax.set_title(key, fontsize=10)
    ax.set_xlabel("score")
    ax.spines[["top", "right"]].set_visible(False)
axes[0][0].set_ylabel("images")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "score_histograms.png", dpi=150)
plt.show()